# NB1 — Trades dataset generator (technical, run once)

Generates a synthetic `trades` dataset with a small, realistic rate of injected data-quality issues, and writes two artifacts for the presentation notebook (NB2):

- **`trades.parquet`** — the dataset to validate
- **`rules.json`** — the rules to run, with their plain-English labels and DuckDB expressions

This notebook is **not** presented. Run it once; then present NB2. Python 3.10.9, DuckDB 1.5.4.

In [1]:
# --- configuration -------------------------------------------------
N_ROWS        = 50_000_000      # lower (e.g. 1_000_000) for a quick regeneration
DEFECT_RATE   = 0.02            # ~2% injected issues per rule, so every check catches something
BUSINESS_DATE = "2025-12-31"    # reference date for the "not in the future" check
SEED          = 0.42            # makes the data reproducible
OUT_DIR       = "."
PARQUET_PATH  = f"{OUT_DIR}/trades.parquet"
RULES_PATH    = f"{OUT_DIR}/rules.json"
DUCKDB_THREADS      = None      # None = use all cores
DUCKDB_MEMORY_LIMIT = None      # include the unit, e.g. "16GB" (a bare number is treated as GB); None = DuckDB default

In [2]:
import duckdb, json, time, os, re
import pyarrow as pa
import numpy as np
con = duckdb.connect()
if DUCKDB_THREADS: con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
def _memlim(v):
    # accept None, '16GB', '16 GB', 16, '16' (bare numbers treated as GB)
    if v is None: return None
    s = str(v).strip().replace(' ', '')
    if not s: return None
    return s + 'GB' if s.replace('.', '', 1).isdigit() else s
_ml = _memlim(DUCKDB_MEMORY_LIMIT)
if _ml: con.execute(f"PRAGMA memory_limit='{_ml}'")
con.execute(f"SELECT setseed({SEED})")
print("DuckDB", duckdb.__version__)

DuckDB 1.5.4


In [3]:
LUHN_OK = "['4539578763621486','6011000990139424','4916338506082832','5500005555555559','4001919257537193']"
def gen_sql(n, d, business_date):
    return f"""CREATE OR REPLACE TABLE trades AS
WITH b0 AS (
  SELECT i,
    CASE WHEN random()<0.002 THEN (floor(random()*1000))::BIGINT ELSE i END AS trade_id,
    (1+(floor(random()*100000))::BIGINT) AS client_id,
    list_element(['XLON','XNYS','XPAR','XETR','XTKS'],1+(floor(random()*5))::INT) AS venue,
    list_element(['EQUITY','BOND','FX','DERIV'],1+(floor(random()*4))::INT) AS asset_class,
    CASE WHEN random()<0.5 THEN 'BUY' ELSE 'SELL' END AS side,
    CASE WHEN random()<{d} THEN -(floor(random()*10))::BIGINT ELSE (1+(floor(random()*100000))::BIGINT) END AS quantity,
    CASE WHEN random()<{d} THEN random()*1000 ELSE round(random()*1000,4) END AS price,
    round(random()*200,2) AS fees, round(random()*150,2) AS tax,
    CASE WHEN random()<{d} THEN 'XXX' WHEN random()<0.55 THEN 'USD' ELSE list_element(['EUR','GBP','JPY','AUD','CAD'],1+(floor(random()*5))::INT) END AS currency,
    CASE WHEN random()<{d} THEN 'FAILED' WHEN random()<0.05 THEN 'CANCELLED' ELSE list_element(['NEW','FILLED','SETTLED','SETTLED'],1+(floor(random()*4))::INT) END AS status,
    (1+(floor(random()*3))::INT) AS pair_idx,
    (DATE '{business_date}' - (floor(random()*365))::INT) AS td_raw
  FROM range({n}) t(i)
),
b1 AS (
  SELECT b0.* EXCLUDE(td_raw),
    CASE WHEN random()<{d} THEN (CASE WHEN isodow(td_raw)>5 THEN td_raw-((isodow(td_raw)-5)::INT) ELSE td_raw END)+(1+(floor(random()*5))::INT)
         WHEN random()<{d} THEN td_raw
         ELSE (CASE WHEN isodow(td_raw)>5 THEN td_raw-((isodow(td_raw)-5)::INT) ELSE td_raw END) END AS trade_date,
    round(abs(quantity)*price,2) AS gross_amount
  FROM b0
)
SELECT
  trade_id, client_id,
  'TRD-'||lpad((i%100000000)::VARCHAR,8,'0') AS parent_order_id,
  CASE WHEN random()<{d} THEN 'BAD'||i::VARCHAR ELSE 'TRD-'||lpad((i%100000000)::VARCHAR,8,'0') END AS trade_ref,
  CASE WHEN random()<{d} THEN 'gb00'||(i%1000)::VARCHAR ELSE 'GB'||lpad((abs(hash(i))%1000000000)::VARCHAR,9,'0')||((i%10))::VARCHAR END AS isin,
  list_element(['AAPL','VOD','HSBA','BP','TSLA','SAP'],1+(floor(random()*6))::INT) AS ticker,
  side, currency, currency AS settlement_ccy, venue, status, asset_class,
  quantity, price, gross_amount, fees, tax,
  CASE WHEN random()<{d} THEN round(gross_amount-fees-tax+5,2) ELSE round(gross_amount-fees-tax,2) END AS net_amount,
  CASE WHEN random()<{d}/2 THEN 'nan'::DOUBLE WHEN random()<{d}/2 THEN 'inf'::DOUBLE WHEN random()<0.001 THEN round(gross_amount*50,2) ELSE gross_amount END AS notional,
  trade_date,
  CASE WHEN status='SETTLED' AND random()<{d} THEN NULL ELSE trade_date+(1+(floor(random()*3))::INT) END AS settlement_date,
  (now() - (floor(random()*36))::INT * INTERVAL 1 HOUR) AS booked_ts,
  CASE WHEN random()<{d} THEN 'LEI'||(i%100)::VARCHAR ELSE 'LEI'||lpad((abs(hash(i))%100000000000000000)::VARCHAR,17,'0') END AS trader_lei,
  CASE WHEN random()<{d} THEN 'UNKNOWN CP '||(i%1000)::VARCHAR ELSE list_element(['GOLDMAN SACHS','GOLDMAN SACH','MORGAN STANLEY','JP MORGAN','BARCLAYS'],1+(floor(random()*5))::INT) END AS counterparty,
  CASE WHEN random()<0.005 THEN round(500+random()*500,2) ELSE round(random()*100,2) END AS risk_score,
  CASE WHEN random()<{d} THEN '{{bad json' WHEN random()<{d} THEN '{{"allocations":[1,2]}}' ELSE '{{"source_system":"'||list_element(['MUREX','CALYPSO','SUMMIT'],1+(floor(random()*3))::INT)||'","allocations":'||list_element(['[1]','[1,2]','[1,2,3,4]'],1+(floor(random()*3))::INT)||'}}' END AS metadata,
  CASE WHEN random()<{d} THEN []::VARCHAR[] WHEN random()<{d} THEN ['EQUITY','EQUITY'] WHEN random()<{d} THEN ['FX'] ELSE ['EQUITY',list_element(['LDN','NY','TOK'],1+(floor(random()*3))::INT)] END AS tags,
  (status='CANCELLED') AS is_cancelled,
  CASE WHEN random()<{d} THEN '80.0,80.0' ELSE round(51.5+random()*0.2,4)::VARCHAR||','||round(0.1+random()*0.2,4)::VARCHAR END AS book_geo,
  list_element(['EMEA','AMER','APAC'],pair_idx) AS region,
  CASE WHEN random()<{d} THEN list_element(['LONDON','NEW YORK','TOKYO'],1+(pair_idx%3)) ELSE list_element(['LONDON','NEW YORK','TOKYO'],pair_idx) END AS sub_region,
  CASE WHEN random()<{d} THEN '4539578763621480' ELSE list_element({LUHN_OK},1+(floor(random()*5))::INT) END AS account_no,
  CASE WHEN random()<{d} THEN '' ELSE (1+(floor(random()*200))::INT)::VARCHAR||' '||list_element(['King St','High Rd','Bank Ave'],1+(floor(random()*3))::INT)||', LONDON' END AS address,
  CASE WHEN random()<{d} THEN 'N/A' ELSE quantity::VARCHAR END AS qty_str,
  CASE WHEN random()<{d} THEN '2024/13/40' ELSE strftime(trade_date,'%Y-%m-%d') END AS date_str,
  CASE WHEN random()<{d} THEN -3 ELSE (floor(random()*40))::INT END AS account_age_yrs,
  CASE WHEN random()<{d} THEN 'Pending review' ELSE list_element(['Order VERIFIED','Amended VERIFIED','Reviewed VERIFIED'],1+(floor(random()*3))::INT) END AS notes,
  CASE WHEN random()<{d} THEN (CASE WHEN side='BUY' THEN round(gross_amount-fees-tax,2) ELSE -round(gross_amount-fees-tax,2) END)
       ELSE (CASE WHEN side='BUY' THEN -round(gross_amount-fees-tax,2) ELSE round(gross_amount-fees-tax,2) END) END AS cash_flow
FROM b1;"""

In [4]:
t=time.perf_counter()
con.execute(gen_sql(N_ROWS, DEFECT_RATE, BUSINESS_DATE))
gen_s=time.perf_counter()-t
t=time.perf_counter()
con.execute(f"COPY trades TO '{PARQUET_PATH}' (FORMAT PARQUET)")
copy_s=time.perf_counter()-t
n_rows=con.execute("SELECT count(*) FROM trades").fetchone()[0]
info=con.execute("PRAGMA table_info('trades')").fetchall()
col_names=[r[1] for r in info]; col_types={r[1]:r[2] for r in info}
size_mb=os.path.getsize(PARQUET_PATH)/1e6
print(f"generated {n_rows:,} rows x {len(col_names)} cols in {gen_s:,.1f}s")
print(f"wrote {PARQUET_PATH} ({size_mb:,.1f} MB) in {copy_s:,.1f}s")

generated 50,000,000 rows x 38 cols in 651.8s
wrote ./trades.parquet (5,805.9 MB) in 301.9s


In [5]:
# custom SQL functions (macros) and vectorised Python (Arrow) UDFs
LUHN_MACRO = "CREATE OR REPLACE MACRO luhn_ok(x) AS (\n x IS NULL OR (length(x)>0 AND (list_reduce(list_transform(range(1,length(x)+1),\n   j -> CASE WHEN ((length(x)-j)%2)=1 THEN (CASE WHEN CAST(x[j] AS INT)*2>9 THEN CAST(x[j] AS INT)*2-9 ELSE CAST(x[j] AS INT)*2 END) ELSE CAST(x[j] AS INT) END),\n   (a,b)->a+b) % 10)=0))"
GEO_MACRO  = "CREATE OR REPLACE MACRO geo_within(g,clat,clon,km) AS (\n g IS NULL OR 6371*2*asin(sqrt(pow(sin((radians(CAST(split_part(g,',',1) AS DOUBLE))-radians(clat))/2),2)\n + cos(radians(clat))*cos(radians(CAST(split_part(g,',',1) AS DOUBLE)))*pow(sin((radians(CAST(split_part(g,',',2) AS DOUBLE))-radians(clon))/2),2)))<=km)"
con.execute(LUHN_MACRO); con.execute(GEO_MACRO)

_addr_re = re.compile(r'^\d+\s+\S.*,\s*\S.+$')
def py_address_valid(addr):
    vals = addr.to_pylist()
    return pa.array([(a is not None and bool(_addr_re.match(a))) for a in vals], type=pa.bool_())

def py_record_anomaly(risk, notional, qty):
    r=np.asarray(risk.to_numpy(zero_copy_only=False),dtype='float64')
    n=np.abs(np.asarray(notional.to_numpy(zero_copy_only=False),dtype='float64'))
    q=np.asarray(qty.to_numpy(zero_copy_only=False),dtype='float64')
    fin=np.isfinite(n)
    anomaly=(r>300)&fin&(n>5e7)&(q>0)
    return pa.array(~anomaly, type=pa.bool_())

con.create_function('py_address_valid', py_address_valid, ['VARCHAR'],'BOOLEAN', type='arrow')
con.create_function('py_record_anomaly', py_record_anomaly, ['DOUBLE','DOUBLE','BIGINT'],'BOOLEAN', type='arrow')
print("registered 2 SQL macros + 2 vectorised Python UDFs")

registered 2 SQL macros + 2 vectorised Python UDFs


In [6]:
RULES = [
  {
    "id": "DS_RECORD_COUNT_BETWEEN",
    "dimension": "Dataset",
    "business_label": "Trade count is within the expected range",
    "columns": [
      "*"
    ],
    "result_kind": "dataset",
    "sql": "count(*) BETWEEN 1 AND 200000000"
  },
  {
    "id": "SCH_FIELDS_SET",
    "dimension": "Schema",
    "business_label": "All expected columns are present",
    "columns": [
      "*"
    ],
    "result_kind": "schema_fields"
  },
  {
    "id": "SCH_FIELD_TYPE",
    "dimension": "Schema",
    "business_label": "Key columns have the expected data types",
    "columns": [
      "notional",
      "trade_date",
      "quantity",
      "trade_id"
    ],
    "result_kind": "schema_types"
  },
  {
    "id": "COMP_NOT_NULL",
    "dimension": "Completeness",
    "business_label": "Trade ID is present",
    "columns": [
      "trade_id"
    ],
    "result_kind": "row",
    "sql": "trade_id IS NOT NULL"
  },
  {
    "id": "COMP_STR_NOT_BLANK",
    "dimension": "Completeness",
    "business_label": "ISIN is present and not blank",
    "columns": [
      "isin"
    ],
    "result_kind": "row",
    "sql": "isin IS NOT NULL AND trim(isin)<>''"
  },
  {
    "id": "COMP_NUM_NOT_NAN",
    "dimension": "Completeness",
    "business_label": "Notional is a real number (no NaN or infinity)",
    "columns": [
      "notional"
    ],
    "result_kind": "row",
    "sql": "notional IS NOT NULL AND isfinite(notional)"
  },
  {
    "id": "VAL_IN_SET_CCY",
    "dimension": "Validity",
    "business_label": "Currency is a known ISO code",
    "columns": [
      "currency"
    ],
    "result_kind": "row",
    "sql": "currency IN ('USD','EUR','GBP','JPY','AUD','CAD')"
  },
  {
    "id": "VAL_IN_SET_STATUS",
    "dimension": "Validity",
    "business_label": "Status is a valid value",
    "columns": [
      "status"
    ],
    "result_kind": "row",
    "sql": "status IN ('NEW','FILLED','SETTLED','CANCELLED')"
  },
  {
    "id": "VAL_IN_SET_VENUE",
    "dimension": "Validity",
    "business_label": "Venue is on the approved list",
    "columns": [
      "venue"
    ],
    "result_kind": "row",
    "sql": "venue IN ('XLON','XNYS','XPAR','XETR','XTKS')"
  },
  {
    "id": "VAL_IN_SET_ASSET",
    "dimension": "Validity",
    "business_label": "Asset class is valid",
    "columns": [
      "asset_class"
    ],
    "result_kind": "row",
    "sql": "asset_class IN ('EQUITY','BOND','FX','DERIV')"
  },
  {
    "id": "VAL_BETWEEN_QTY",
    "dimension": "Validity",
    "business_label": "Quantity is a sensible amount",
    "columns": [
      "quantity"
    ],
    "result_kind": "row",
    "sql": "quantity BETWEEN 1 AND 10000000"
  },
  {
    "id": "VAL_MAX_DECIMAL_PLACES",
    "dimension": "Validity",
    "business_label": "Price is within tick precision (<=4 dp)",
    "columns": [
      "price"
    ],
    "result_kind": "row",
    "sql": "price IS NULL OR round(price,4)=price"
  },
  {
    "id": "VAL_LENGTH_BETWEEN_LEI",
    "dimension": "Validity",
    "business_label": "Trader LEI is 20 characters",
    "columns": [
      "trader_lei"
    ],
    "result_kind": "row",
    "sql": "trader_lei IS NULL OR length(trader_lei)=20"
  },
  {
    "id": "VAL_REGEX_MATCH_ISIN",
    "dimension": "Validity",
    "business_label": "ISIN has a valid format",
    "columns": [
      "isin"
    ],
    "result_kind": "row",
    "sql": "isin IS NULL OR regexp_matches(isin,'^[A-Z]{2}[A-Z0-9]{9}[0-9]$')"
  },
  {
    "id": "VAL_LIKE_REF",
    "dimension": "Validity",
    "business_label": "Trade reference matches TRD- pattern",
    "columns": [
      "trade_ref"
    ],
    "result_kind": "row",
    "sql": "trade_ref LIKE 'TRD-%'"
  },
  {
    "id": "VAL_ILIKE_REF",
    "dimension": "Validity",
    "business_label": "Trade reference matches pattern (case-insensitive)",
    "columns": [
      "trade_ref"
    ],
    "result_kind": "row",
    "sql": "trade_ref ILIKE 'trd-%'"
  },
  {
    "id": "VAL_STARTS_WITH_ISIN",
    "dimension": "Validity",
    "business_label": "ISIN starts with a country code",
    "columns": [
      "isin"
    ],
    "result_kind": "row",
    "sql": "starts_with(isin,'GB')"
  },
  {
    "id": "VAL_CONTAINS_NOTES",
    "dimension": "Validity",
    "business_label": "Notes contain a verification marker",
    "columns": [
      "notes"
    ],
    "result_kind": "row",
    "sql": "contains(notes,'VERIFIED')"
  },
  {
    "id": "VAL_FUZZY_MATCH_CP",
    "dimension": "Validity",
    "business_label": "Counterparty matches a known name (typo-tolerant)",
    "columns": [
      "counterparty"
    ],
    "result_kind": "row",
    "sql": "list_min(list_transform(['GOLDMAN SACHS','MORGAN STANLEY','JP MORGAN','BARCLAYS'], n -> levenshtein(upper(counterparty),n)))<=2"
  },
  {
    "id": "VAL_TYPE_PARSEABLE_QTY",
    "dimension": "Validity",
    "business_label": "Quantity string parses as a number",
    "columns": [
      "qty_str"
    ],
    "result_kind": "row",
    "sql": "try_cast(qty_str AS BIGINT) IS NOT NULL"
  },
  {
    "id": "VAL_DATE_FORMAT",
    "dimension": "Validity",
    "business_label": "Date string is a valid date",
    "columns": [
      "date_str"
    ],
    "result_kind": "row",
    "sql": "try_strptime(date_str,'%Y-%m-%d') IS NOT NULL"
  },
  {
    "id": "VAL_AGE_BETWEEN",
    "dimension": "Validity",
    "business_label": "Account age is plausible",
    "columns": [
      "account_age_yrs"
    ],
    "result_kind": "row",
    "sql": "account_age_yrs BETWEEN 0 AND 100"
  },
  {
    "id": "VAL_DOW_IN_SET",
    "dimension": "Validity",
    "business_label": "Trade booked on a business day",
    "columns": [
      "trade_date"
    ],
    "result_kind": "row",
    "sql": "isodow(trade_date) BETWEEN 1 AND 5"
  },
  {
    "id": "VAL_NOT_FUTURE",
    "dimension": "Validity",
    "business_label": "Trade date is not in the future",
    "columns": [
      "trade_date"
    ],
    "result_kind": "row",
    "sql": "trade_date <= DATE '2025-12-31'"
  },
  {
    "id": "VAL_BOOL_IS_FALSE",
    "dimension": "Validity",
    "business_label": "Trade is not flagged cancelled",
    "columns": [
      "is_cancelled"
    ],
    "result_kind": "row",
    "sql": "is_cancelled = FALSE"
  },
  {
    "id": "VAL_JSON_PARSEABLE",
    "dimension": "Validity",
    "business_label": "Metadata is valid JSON",
    "columns": [
      "metadata"
    ],
    "result_kind": "row",
    "sql": "json_valid(metadata)"
  },
  {
    "id": "VAL_JSON_HAS_KEY",
    "dimension": "Validity",
    "business_label": "Metadata has a source system",
    "columns": [
      "metadata"
    ],
    "result_kind": "row",
    "sql": "NOT json_valid(metadata) OR json_extract_string(CASE WHEN json_valid(metadata) THEN metadata ELSE '{}' END,'$.source_system') IS NOT NULL"
  },
  {
    "id": "VAL_JSON_ARRAY_LEN",
    "dimension": "Validity",
    "business_label": "Allocation array is a sensible size",
    "columns": [
      "metadata"
    ],
    "result_kind": "row",
    "sql": "NOT json_valid(metadata) OR coalesce(json_array_length(json_extract(CASE WHEN json_valid(metadata) THEN metadata ELSE '{}' END,'$.allocations')),0) BETWEEN 1 AND 50"
  },
  {
    "id": "LIST_CONTAINS",
    "dimension": "Validity",
    "business_label": "Tags include a required tag",
    "columns": [
      "tags"
    ],
    "result_kind": "row",
    "sql": "list_contains(tags,'EQUITY')"
  },
  {
    "id": "LIST_LENGTH_BETWEEN",
    "dimension": "Validity",
    "business_label": "Tag count is within range",
    "columns": [
      "tags"
    ],
    "result_kind": "row",
    "sql": "len(tags) BETWEEN 1 AND 10"
  },
  {
    "id": "LIST_NO_DUPLICATES",
    "dimension": "Validity",
    "business_label": "No repeated tags",
    "columns": [
      "tags"
    ],
    "result_kind": "row",
    "sql": "len(tags)=len(list_distinct(tags))"
  },
  {
    "id": "COND_IF_THEN",
    "dimension": "Consistency",
    "business_label": "If settled, a settlement date is present",
    "columns": [
      "status",
      "settlement_date"
    ],
    "result_kind": "row",
    "sql": "status<>'SETTLED' OR settlement_date IS NOT NULL"
  },
  {
    "id": "MULT_FIELD_EXPR_EQ",
    "dimension": "Consistency",
    "business_label": "Net equals gross minus fees and tax (tolerance)",
    "columns": [
      "net_amount",
      "gross_amount",
      "fees",
      "tax"
    ],
    "result_kind": "row",
    "sql": "abs(net_amount-(gross_amount-fees-tax))<=0.01"
  },
  {
    "id": "MULT_FIELD_COMPARE_GT",
    "dimension": "Consistency",
    "business_label": "Settlement date is on or after trade date",
    "columns": [
      "settlement_date",
      "trade_date"
    ],
    "result_kind": "row",
    "sql": "settlement_date IS NULL OR settlement_date>=trade_date"
  },
  {
    "id": "MULT_FIELD_BOTH_NULL",
    "dimension": "Consistency",
    "business_label": "Counterparty and LEI are both present or both absent",
    "columns": [
      "counterparty",
      "trader_lei"
    ],
    "result_kind": "row",
    "sql": "(counterparty IS NULL)=(trader_lei IS NULL)"
  },
  {
    "id": "MULT_FIELD_COMB_IN_SET",
    "dimension": "Consistency",
    "business_label": "Region and sub-region are a valid pair",
    "columns": [
      "region",
      "sub_region"
    ],
    "result_kind": "row",
    "sql": "(region,sub_region) IN (('EMEA','LONDON'),('AMER','NEW YORK'),('APAC','TOKYO'))"
  },
  {
    "id": "MULT_FIELD_SIGN",
    "dimension": "Consistency",
    "business_label": "Buy/sell agrees with the cash-flow sign",
    "columns": [
      "side",
      "cash_flow"
    ],
    "result_kind": "row",
    "sql": "(side='BUY' AND cash_flow<=0) OR (side='SELL' AND cash_flow>=0)"
  },
  {
    "id": "STAT_OUTLIER_ZSCORE",
    "dimension": "Validity",
    "business_label": "Risk score is not a statistical outlier (>3 sigma)",
    "columns": [
      "risk_score"
    ],
    "result_kind": "row",
    "sql": "abs(risk_score-(SELECT avg(risk_score) FROM {src}))<=3*(SELECT stddev_pop(risk_score) FROM {src})"
  },
  {
    "id": "STAT_EXTREME_P99",
    "dimension": "Validity",
    "business_label": "Notional is not an extreme value (above p99)",
    "columns": [
      "notional"
    ],
    "result_kind": "row",
    "sql": "notional IS NULL OR NOT isfinite(notional) OR notional <= (SELECT approx_quantile(notional,0.99) FROM {src} WHERE isfinite(notional))"
  },
  {
    "id": "CUSTOM_SQL_CHECKSUM",
    "dimension": "Validity",
    "business_label": "Account number passes a checksum (Luhn)",
    "columns": [
      "account_no"
    ],
    "result_kind": "row",
    "sql": "luhn_ok(account_no)",
    "requires": [
      "luhn"
    ]
  },
  {
    "id": "CUSTOM_SQL_GEO_WITHIN",
    "dimension": "Validity",
    "business_label": "Booking origin is within the allowed region",
    "columns": [
      "book_geo"
    ],
    "result_kind": "row",
    "sql": "geo_within(book_geo,51.5,0.12,50)",
    "requires": [
      "geo"
    ]
  },
  {
    "id": "CUSTOM_PY_ADDRESS_VALID",
    "dimension": "Validity",
    "business_label": "Counterparty address is well formed",
    "columns": [
      "address"
    ],
    "result_kind": "row",
    "sql": "py_address_valid(address)",
    "requires": [
      "py_address_valid"
    ]
  },
  {
    "id": "CUSTOM_PY_RECORD_ANOMALY",
    "dimension": "Validity",
    "business_label": "No cross-field trade anomaly",
    "columns": [
      "risk_score",
      "notional",
      "quantity"
    ],
    "result_kind": "row",
    "sql": "py_record_anomaly(risk_score, notional, quantity)",
    "requires": [
      "py_record_anomaly"
    ]
  },
  {
    "id": "STAT_MEAN_BETWEEN",
    "dimension": "Validity",
    "business_label": "Average notional is in a sane band",
    "columns": [
      "notional"
    ],
    "result_kind": "aggregate",
    "sql": "avg(CASE WHEN isfinite(notional) THEN notional END) BETWEEN 0 AND 1e9"
  },
  {
    "id": "STAT_MIN_BETWEEN",
    "dimension": "Validity",
    "business_label": "Minimum notional is in range",
    "columns": [
      "notional"
    ],
    "result_kind": "aggregate",
    "sql": "min(CASE WHEN isfinite(notional) THEN notional END) BETWEEN 0 AND 1e9"
  },
  {
    "id": "STAT_MAX_BETWEEN",
    "dimension": "Validity",
    "business_label": "Maximum notional is in range",
    "columns": [
      "notional"
    ],
    "result_kind": "aggregate",
    "sql": "max(CASE WHEN isfinite(notional) THEN notional END) BETWEEN 0 AND 1e12"
  },
  {
    "id": "STAT_SUM_BETWEEN",
    "dimension": "Validity",
    "business_label": "Total notional is in the expected range",
    "columns": [
      "notional"
    ],
    "result_kind": "aggregate",
    "sql": "sum(CASE WHEN isfinite(notional) THEN notional END) BETWEEN 0 AND 1e18"
  },
  {
    "id": "STAT_MODE_IN_SET",
    "dimension": "Validity",
    "business_label": "Most common currency is expected",
    "columns": [
      "currency"
    ],
    "result_kind": "aggregate",
    "sql": "mode(currency) IN ('USD','EUR','GBP')"
  },
  {
    "id": "MULT_FIELD_CORRELATION",
    "dimension": "Consistency",
    "business_label": "Quantity and notional move together",
    "columns": [
      "quantity",
      "notional"
    ],
    "result_kind": "aggregate",
    "sql": "corr(quantity, CASE WHEN isfinite(notional) THEN notional END) BETWEEN 0.05 AND 1.0"
  },
  {
    "id": "TIMELINESS_FRESHNESS",
    "dimension": "Timeliness",
    "business_label": "The feed is fresh (booked within the last week)",
    "columns": [
      "booked_ts"
    ],
    "result_kind": "aggregate",
    "sql": "max(booked_ts) >= now() - INTERVAL 7 DAY"
  },
  {
    "id": "UNIQ_DUPLICATES_EXACT",
    "dimension": "Uniqueness",
    "business_label": "No duplicate trade IDs",
    "columns": [
      "trade_id"
    ],
    "result_kind": "dedup",
    "variant": "exact"
  },
  {
    "id": "UNIQ_DUPLICATES_APPROX",
    "dimension": "Uniqueness",
    "business_label": "Heavy-duplication screen on trade ID (fast estimate)",
    "columns": [
      "trade_id"
    ],
    "result_kind": "dedup",
    "variant": "approx"
  },
  {
    "id": "MULT_FIELD_DUPLICATES_EXACT",
    "dimension": "Uniqueness",
    "business_label": "Same trade ID booked more than once on a date",
    "columns": [
      "trade_id",
      "trade_date"
    ],
    "result_kind": "dedup",
    "variant": "exact"
  },
  {
    "id": "MULT_FIELD_DUPLICATES_APPROX",
    "dimension": "Uniqueness",
    "business_label": "Heavy-duplication screen on trade ID and date (fast estimate)",
    "columns": [
      "trade_id",
      "trade_date"
    ],
    "result_kind": "dedup",
    "variant": "approx"
  }
]

# fill schema expectations from the generated table
for r in RULES:
    if r["result_kind"]=="schema_fields": r["expected"]=col_names
    if r["result_kind"]=="schema_types":  r["expected"]={c:col_types[c] for c in r["columns"]}

rules_doc = {
  "dataset":"trades", "parquet_path":PARQUET_PATH, "business_date":BUSINESS_DATE,
  "generated_rows": n_rows,
  "setup": {"sql_macros":[LUHN_MACRO, GEO_MACRO],
            "python_udfs":[{"name":"py_address_valid","params":["VARCHAR"],"returns":"BOOLEAN"},
                           {"name":"py_record_anomaly","params":["DOUBLE","DOUBLE","BIGINT"],"returns":"BOOLEAN"}]},
  "rules": RULES }
json.dump(rules_doc, open(RULES_PATH,"w"), indent=2)
print(f"wrote {RULES_PATH}: {len(RULES)} rules across {len(set(r['dimension'] for r in RULES))} dimensions")

wrote ./rules.json: 54 rules across 7 dimensions


In [7]:
# --- technical sanity: confirm every rule executes and most catch something ---
SRC = "trades"
def sub(s): return s.replace("{src}", SRC)
row = [r for r in RULES if r["result_kind"] == "row"]
agg = [r for r in RULES if r["result_kind"] == "aggregate"]
ded = [r for r in RULES if r["result_kind"] == "dedup"]
total = con.execute(f"SELECT count(*) FROM {SRC}").fetchone()[0]

# one combined scan for every row-level rule
row_sel = ", ".join(f'count_if(NOT ({sub(r["sql"])})) AS c{i}' for i, r in enumerate(row))
t = time.perf_counter()
rres = con.execute(f"SELECT {row_sel} FROM {SRC}").fetchone()
row_s = time.perf_counter() - t

# one pass for aggregate rules
agg_sel = ", ".join(f'({r["sql"]}) AS a{i}' for i, r in enumerate(agg))
ares = con.execute(f"SELECT {agg_sel} FROM {SRC}").fetchone()

# dedup rules
def dcount(cols, variant):
    key = ", ".join(cols)
    if variant == "exact":
        distinct = con.execute(f"SELECT count(*) FROM (SELECT DISTINCT {key} FROM {SRC})").fetchone()[0]
        return total - distinct
    expr = cols[0] if len(cols) == 1 else "concat_ws('|', " + ", ".join(f"{c}::VARCHAR" for c in cols) + ")"
    ap = con.execute(f"SELECT approx_count_distinct({expr}) FROM {SRC}").fetchone()[0]
    return total - ap

print(f"{total:,} rows | row-rule scan {row_s:,.2f}s\n")
print(f"{'dimension':<13}{'rule':<30}{'result':>12}")
print("-" * 57)
for i, r in enumerate(row): print(f"{r['dimension']:<13}{r['id']:<30}{rres[i]:>10,} !")
for i, r in enumerate(agg): print(f"{r['dimension']:<13}{r['id']:<30}{('PASS' if ares[i] else 'FAIL'):>12}")
for r in ded:               print(f"{r['dimension']:<13}{r['id']:<30}{dcount(r['columns'], r['variant']):>10,} !")
fired = sum(1 for x in rres if x > 0)
print(f"\nrow rules firing: {fired}/{len(row)}   (\"!\" = rows caught; a few clean-pass rules are expected)")

50,000,000 rows | row-rule scan 689.58s

dimension    rule                                result
---------------------------------------------------------
Completeness COMP_NOT_NULL                          0 !
Completeness COMP_STR_NOT_BLANK                     0 !
Completeness COMP_NUM_NOT_NAN                 994,949 !
Validity     VAL_IN_SET_CCY                   999,235 !
Validity     VAL_IN_SET_STATUS              1,001,190 !
Validity     VAL_IN_SET_VENUE                       0 !
Validity     VAL_IN_SET_ASSET                       0 !
Validity     VAL_BETWEEN_QTY                1,000,867 !
Validity     VAL_MAX_DECIMAL_PLACES         1,002,055 !
Validity     VAL_LENGTH_BETWEEN_LEI         1,000,227 !
Validity     VAL_REGEX_MATCH_ISIN           1,000,029 !
Validity     VAL_LIKE_REF                     998,004 !
Validity     VAL_ILIKE_REF                    998,004 !
Validity     VAL_STARTS_WITH_ISIN           1,000,029 !
Validity     VAL_CONTAINS_NOTES               999,332 !
Valid

In [8]:
print("="*60)
print(f"NB1 complete.")
print(f"  dataset : {PARQUET_PATH}  ({n_rows:,} rows, {len(col_names)} cols, {size_mb:,.1f} MB)")
print(f"  rules   : {RULES_PATH}  ({len(RULES)} rules)")
print(f"  next    : run NB2 to preview + validate.")
print("="*60)

NB1 complete.
  dataset : ./trades.parquet  (50,000,000 rows, 38 cols, 5,805.9 MB)
  rules   : ./rules.json  (54 rules)
  next    : run NB2 to preview + validate.
